# Лабораторная работа № 4
## Проектирование и сборка аналитической витрины

**Проект:** Seoul Bike Sharing Demand  
**Группа:** ИВТ-241

### Аналитический вопрос

В какие часы, сезоны и типы дней наблюдается наиболее высокий и низкий спрос на аренду велосипедов?

### Цель работы

Спроектировать и собрать аналитическую витрину на основе подготовленных данных
из слоя `staging`, сохранить её в слое `mart` в S3-хранилище и проверить
корректность сформированных данных.

### Источник данных

Подготовленный файл `staging.parquet` из лабораторной работы № 3,
расположенный в S3-хранилище SeaweedFS.

### Планируемый результат

Витрина `hourly_demand`, в которой одна строка представляет одно
почасовое наблюдение за спросом на аренду велосипедов.

In [2]:
from pathlib import Path
import os

import duckdb
from dotenv import load_dotenv

# Находим корень проекта и загружаем локальные настройки.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(
    (path for path in candidates if (path / ".env").is_file()),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Не найден .env в папке проекта.")

load_dotenv(PROJECT_ROOT / ".env")

# Дата загрузки соответствует staging из ЛР № 3.
INGESTED_ON = "2026-10-13"
DATASET_SLUG = "seoul_bike"

# Входная витрина и путь для результата.
STAGING_S3_PATH = (
    f"s3://staging/{DATASET_SLUG}/"
    f"source_ingested_on={INGESTED_ON}/staging.parquet"
)

MART_S3_PATH = (
    f"s3://mart/{DATASET_SLUG}/"
    f"source_ingested_on={INGESTED_ON}/hourly_demand.parquet"
)

# DuckDB использует эти выражения как источники в SQL-запросах.
staging_file = (
    f"read_parquet('{STAGING_S3_PATH}', hive_partitioning=true)"
)

# Проверяем наличие настроек S3.
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "localhost:8333")
S3_ACCESS_KEY = os.getenv("S3_ACCESS_KEY")
S3_SECRET_KEY = os.getenv("S3_SECRET_KEY")

if not S3_ACCESS_KEY or not S3_SECRET_KEY:
    raise RuntimeError(
        "В .env должны быть заданы S3_ACCESS_KEY и S3_SECRET_KEY."
    )

# Подключаем DuckDB к SeaweedFS.
con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

s3_settings = {
    "s3_endpoint": S3_ENDPOINT,
    "s3_access_key_id": S3_ACCESS_KEY,
    "s3_secret_access_key": S3_SECRET_KEY,
    "s3_region": "us-east-1",
}

for name, value in s3_settings.items():
    escaped_value = value.replace("'", "''")
    con.execute(f"SET {name} = '{escaped_value}';")

con.execute("SET s3_use_ssl = false;")
con.execute("SET s3_url_style = 'path';")

# Проверяем чтение staging.parquet непосредственно из S3.
staging_rows = con.execute(
    f"SELECT COUNT(*) FROM {staging_file}"
).fetchone()[0]

staging_schema = con.execute(
    f"DESCRIBE SELECT * FROM {staging_file}"
).fetchall()

print("Корень проекта:", PROJECT_ROOT)
print("Входной staging:", STAGING_S3_PATH)
print("Выходной mart:", MART_S3_PATH)
print("DuckDB:", duckdb.__version__)
print("Количество строк в staging:", staging_rows)

print("\nСхема staging:")
for column in staging_schema:
    print(f"  {column[0]}: {column[1]}")

Корень проекта: c:\Users\Даниил\Desktop\Учусь типа\Семестр 5\ИТАД\SeoulBikeLab2
Входной staging: s3://staging/seoul_bike/source_ingested_on=2026-10-13/staging.parquet
Выходной mart: s3://mart/seoul_bike/source_ingested_on=2026-10-13/hourly_demand.parquet
DuckDB: 1.2.0
Количество строк в staging: 8760

Схема staging:
  observation_date: DATE
  rented_bike_count: BIGINT
  hour: BIGINT
  temperature_c: DOUBLE
  humidity_pct: BIGINT
  wind_speed_m_s: DOUBLE
  visibility_10m: BIGINT
  dew_point_temperature_c: DOUBLE
  solar_radiation_mj_m2: DOUBLE
  rainfall_mm: DOUBLE
  snowfall_cm: DOUBLE
  season: VARCHAR
  holiday: VARCHAR
  functioning_day: BOOLEAN
  ingested_on: DATE
  quality_flag_humidity_out_of_range: BOOLEAN
  source_ingested_on: DATE


In [3]:

# Предварительная проверка staging перед сборкой витрины.

profile = con.execute(
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT observation_date) AS date_count,
        MIN(observation_date) AS min_date,
        MAX(observation_date) AS max_date,
        MIN(hour) AS min_hour,
        MAX(hour) AS max_hour,
        MIN(rented_bike_count) AS min_rentals,
        MAX(rented_bike_count) AS max_rentals,
        COUNT(*) FILTER (
            WHERE observation_date IS NULL
               OR hour IS NULL
               OR rented_bike_count IS NULL
        ) AS missing_core_rows
    FROM {staging_file}
    """
).fetchone()

print("Профиль входных данных")
print("Строк:", profile[0])
print("Уникальных дат:", profile[1])
print("Первая дата:", profile[2])
print("Последняя дата:", profile[3])
print("Минимальный час:", profile[4])
print("Максимальный час:", profile[5])
print("Минимум аренд:", profile[6])
print("Максимум аренд:", profile[7])
print("Строк с пропусками ключевых полей:", profile[8])

# Проверяем повторы по составному ключу.
duplicate_keys = con.execute(
    f"""
    SELECT COUNT(*)
    FROM (
        SELECT observation_date, hour
        FROM {staging_file}
        GROUP BY observation_date, hour
        HAVING COUNT(*) > 1
    ) duplicates
    """
).fetchone()[0]

print("Повторяющихся ключей дата + час:", duplicate_keys)

# Проверяем доступные сезоны.
print("\nСезоны:")
for season, count in con.execute(
    f"""
    SELECT season, COUNT(*)
    FROM {staging_file}
    GROUP BY season
    ORDER BY season
    """
).fetchall():
    print(f"  {season}: {count}")

# Проверяем типы календарных дней.
print("\nПраздничный статус:")
for holiday, count in con.execute(
    f"""
    SELECT holiday, COUNT(*)
    FROM {staging_file}
    GROUP BY holiday
    ORDER BY holiday
    """
).fetchall():
    print(f"  {holiday}: {count}")

print("\nРабота системы:")
for functioning, count in con.execute(
    f"""
    SELECT functioning_day, COUNT(*)
    FROM {staging_file}
    GROUP BY functioning_day
    ORDER BY functioning_day
    """
).fetchall():
    print(f"  {functioning}: {count}")


Профиль входных данных
Строк: 8760
Уникальных дат: 365
Первая дата: 2017-12-01
Последняя дата: 2018-11-30
Минимальный час: 0
Максимальный час: 23
Минимум аренд: 0
Максимум аренд: 3556
Строк с пропусками ключевых полей: 0
Повторяющихся ключей дата + час: 0

Сезоны:
  Autumn: 2184
  Spring: 2208
  Summer: 2208
  Winter: 2160

Праздничный статус:
  Holiday: 432
  No Holiday: 8328

Работа системы:
  False: 295
  True: 8465


In [4]:

# Формируем аналитическую витрину hourly_demand.
# Одна строка представляет одно почасовое наблюдение.

con.execute(f"""
CREATE OR REPLACE TEMP VIEW mart_hourly_demand AS
SELECT
    observation_date,
    hour,
    rented_bike_count,

    season,
    holiday,
    functioning_day,

    CAST(
        EXTRACT(ISODOW FROM observation_date)
        AS INTEGER
    ) AS iso_weekday,

    strftime(observation_date, '%A') AS day_of_week,

    CASE
        WHEN EXTRACT(ISODOW FROM observation_date) IN (6, 7)
            THEN 'Weekend'
        ELSE 'Weekday'
    END AS day_type,

    temperature_c,
    humidity_pct,
    wind_speed_m_s,
    visibility_10m,
    dew_point_temperature_c,
    solar_radiation_mj_m2,
    rainfall_mm,
    snowfall_cm,

    quality_flag_humidity_out_of_range

FROM {staging_file}
""")

# Проверяем число строк и основные показатели сформированной витрины.
mart_profile = con.execute("""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT observation_date) AS date_count,
    COUNT(*) FILTER (
        WHERE observation_date IS NULL
           OR hour IS NULL
           OR rented_bike_count IS NULL
    ) AS missing_core_rows,
    MIN(rented_bike_count) AS min_rentals,
    MAX(rented_bike_count) AS max_rentals,
    MIN(iso_weekday) AS min_iso_weekday,
    MAX(iso_weekday) AS max_iso_weekday
FROM mart_hourly_demand
""").fetchone()

input_rows = con.execute(
    f"SELECT COUNT(*) FROM {staging_file}"
).fetchone()[0]

print("Проверка временной витрины hourly_demand")
print("Строк во входном staging:", input_rows)
print("Строк в витрине:", mart_profile[0])
print("Уникальных дат:", mart_profile[1])
print("Строк с пропусками ключевых полей:", mart_profile[2])
print("Минимум аренд:", mart_profile[3])
print("Максимум аренд:", mart_profile[4])
print("Минимальный ISO-день недели:", mart_profile[5])
print("Максимальный ISO-день недели:", mart_profile[6])

assert mart_profile[0] == input_rows, (
    "Число строк витрины не совпадает с входным staging."
)

assert mart_profile[2] == 0, (
    "В витрине обнаружены пропуски в ключевых полях."
)

assert mart_profile[5] == 1 and mart_profile[6] == 7, (
    "Некорректный диапазон ISO-дней недели."
)

print("\nПроверки временной витрины пройдены.")


Проверка временной витрины hourly_demand
Строк во входном staging: 8760
Строк в витрине: 8760
Уникальных дат: 365
Строк с пропусками ключевых полей: 0
Минимум аренд: 0
Максимум аренд: 3556
Минимальный ISO-день недели: 1
Максимальный ISO-день недели: 7

Проверки временной витрины пройдены.


In [5]:

# Сохраняем витрину в S3 в формате Parquet.
# При повторном запуске обновляем существующий объект.

escaped_mart_path = MART_S3_PATH.replace("'", "''")

con.execute(f"""
COPY (
    SELECT *
    FROM mart_hourly_demand
)
TO '{escaped_mart_path}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD,
    OVERWRITE_OR_IGNORE TRUE
)
""")

print("Витрина сохранена.")
print("Путь:", MART_S3_PATH)
print("Формат: Parquet")


Витрина сохранена.
Путь: s3://mart/seoul_bike/source_ingested_on=2026-10-13/hourly_demand.parquet
Формат: Parquet


In [6]:
# Повторно читаем сохранённую витрину непосредственно из S3.

escaped_mart_path = MART_S3_PATH.replace("'", "''")

mart_file = (
    f"read_parquet('{escaped_mart_path}', hive_partitioning=true)"
)

# Сравниваем число строк во входном staging и в сохранённой витрине.
staging_rows = con.execute(
    f"SELECT COUNT(*) FROM {staging_file}"
).fetchone()[0]

mart_rows = con.execute(
    f"SELECT COUNT(*) FROM {mart_file}"
).fetchone()[0]

print("Проверка сохранённой витрины")
print("Строк во входном staging:", staging_rows)
print("Строк в сохранённом mart:", mart_rows)

# Проверяем схему Parquet, прочитанного из S3.
print("\nСхема сохранённой витрины:")
mart_schema = con.execute(
    f"DESCRIBE SELECT * FROM {mart_file}"
).fetchall()

for column in mart_schema:
    print(f"  {column[0]}: {column[1]}")

# Выполняем проверки ключа, значений и флагов качества.
mart_validation = con.execute(
    f"""
    WITH checked AS (
        SELECT
            *,
            COUNT(*) OVER (
                PARTITION BY observation_date, hour
            ) AS key_count
        FROM {mart_file}
    )
    SELECT
        COUNT(*) FILTER (
            WHERE key_count > 1
        ) AS duplicate_key_rows,

        COUNT(*) FILTER (
            WHERE observation_date IS NULL
               OR hour IS NULL
               OR rented_bike_count IS NULL
        ) AS missing_core_rows,

        COUNT(*) FILTER (
            WHERE hour IS NULL OR hour < 0 OR hour > 23
        ) AS invalid_hour_rows,

        COUNT(*) FILTER (
            WHERE rented_bike_count < 0
        ) AS negative_rental_rows,

        COUNT(*) FILTER (
            WHERE iso_weekday IS NULL
               OR iso_weekday < 1
               OR iso_weekday > 7
        ) AS invalid_weekday_rows,

        COUNT(*) FILTER (
            WHERE day_type IS NULL
               OR day_type NOT IN ('Weekday', 'Weekend')
        ) AS invalid_day_type_rows,

        COUNT(*) FILTER (
            WHERE functioning_day = FALSE
              AND rented_bike_count <> 0
        ) AS inconsistent_functioning_rows,

        COUNT(*) FILTER (
            WHERE quality_flag_humidity_out_of_range
                IS DISTINCT FROM (
                    humidity_pct IS NULL
                    OR humidity_pct < 0
                    OR humidity_pct > 100
                )
        ) AS humidity_flag_mismatch_rows

    FROM checked
    """
).fetchone()

validation_names = [
    "Повторяющиеся ключи дата + час",
    "Пропуски ключевых полей",
    "Недопустимые часы",
    "Отрицательное количество аренд",
    "Недопустимые дни недели",
    "Недопустимые типы дней",
    "Аренды при неработающей системе",
    "Несоответствия флага влажности",
]

print("\nРезультаты проверок:")
for name, count in zip(validation_names, mart_validation):
    print(f"{name}: {count}")

# Проверяем сохранность флагов влажности относительно staging.
staging_flag_count = con.execute(
    f"""
    SELECT COUNT(*)
    FROM {staging_file}
    WHERE quality_flag_humidity_out_of_range
    """
).fetchone()[0]

mart_flag_count = con.execute(
    f"""
    SELECT COUNT(*)
    FROM {mart_file}
    WHERE quality_flag_humidity_out_of_range
    """
).fetchone()[0]

print("\nФлаги влажности в staging:", staging_flag_count)
print("Флаги влажности в mart:", mart_flag_count)

# Итоговые проверки.
assert mart_rows == staging_rows, (
    "Число строк mart не совпадает с staging."
)

assert all(count == 0 for count in mart_validation), (
    "Обнаружены нарушения в сохранённой витрине."
)

assert mart_flag_count == staging_flag_count, (
    "Количество флагов влажности изменилось при сборке витрины."
)

print("\nВсе проверки сохранённой витрины пройдены.")

Проверка сохранённой витрины
Строк во входном staging: 8760
Строк в сохранённом mart: 8760

Схема сохранённой витрины:
  observation_date: DATE
  hour: BIGINT
  rented_bike_count: BIGINT
  season: VARCHAR
  holiday: VARCHAR
  functioning_day: BOOLEAN
  iso_weekday: INTEGER
  day_of_week: VARCHAR
  day_type: VARCHAR
  temperature_c: DOUBLE
  humidity_pct: BIGINT
  wind_speed_m_s: DOUBLE
  visibility_10m: BIGINT
  dew_point_temperature_c: DOUBLE
  solar_radiation_mj_m2: DOUBLE
  rainfall_mm: DOUBLE
  snowfall_cm: DOUBLE
  quality_flag_humidity_out_of_range: BOOLEAN
  source_ingested_on: DATE

Результаты проверок:
Повторяющиеся ключи дата + час: 0
Пропуски ключевых полей: 0
Недопустимые часы: 0
Отрицательное количество аренд: 0
Недопустимые дни недели: 0
Недопустимые типы дней: 0
Аренды при неработающей системе: 0
Несоответствия флага влажности: 0

Флаги влажности в staging: 0
Флаги влажности в mart: 0

Все проверки сохранённой витрины пройдены.


In [7]:

# Анализ спроса по сезонам, типам дней и часам.
# Средний спрос рассчитывается для наблюдений,
# когда система проката работала.

demand_by_slice = con.execute(
    f"""
    SELECT
        season,
        day_type,
        hour,
        COUNT(*) AS observation_count,
        COUNT(*) FILTER (
            WHERE functioning_day
        ) AS operating_observation_count,
        ROUND(
            AVG(rented_bike_count) FILTER (
                WHERE functioning_day
            ),
            2
        ) AS avg_rentals_operating
    FROM {mart_file}
    GROUP BY season, day_type, hour
    HAVING COUNT(*) FILTER (
        WHERE functioning_day
    ) > 0
    """
).fetchall()

# Показываем 10 срезов с самым высоким и самым низким
# средним количеством аренд в часы работы системы.

highest_demand = sorted(
    demand_by_slice,
    key=lambda row: row[5],
    reverse=True,
)[:10]

lowest_demand = sorted(
    demand_by_slice,
    key=lambda row: row[5],
)[:10]

def print_demand_table(title, rows):
    print(f"\n{title}")
    print(
        f"{'Сезон':<10} {'Тип дня':<10} {'Час':>4} "
        f"{'Наблюдений':>12} {'Работала система':>17} "
        f"{'Среднее аренд':>14}"
    )

    for season, day_type, hour, count, operating_count, avg_rentals in rows:
        print(
            f"{season:<10} {day_type:<10} {hour:>4} "
            f"{count:>12} {operating_count:>17} "
            f"{avg_rentals:>14.2f}"
        )

print_demand_table("10 срезов с наибольшим средним спросом", highest_demand)
print_demand_table("10 срезов с наименьшим средним спросом", lowest_demand)



10 срезов с наибольшим средним спросом
Сезон      Тип дня     Час   Наблюдений  Работала система  Среднее аренд
Summer     Weekday      18           66                66        2327.36
Autumn     Weekday      18           65                57        2200.95
Summer     Weekday      19           66                66        1950.52
Spring     Weekday      18           66                64        1887.81
Summer     Weekday      20           66                66        1818.03
Summer     Weekday      21           66                66        1785.83
Summer     Weekend      20           26                26        1761.04
Summer     Weekend      19           26                26        1733.73
Summer     Weekday       8           66                66        1720.80
Autumn     Weekday       8           65                57        1688.11

10 срезов с наименьшим средним спросом
Сезон      Тип дня     Час   Наблюдений  Работала система  Среднее аренд
Winter     Weekend       5           26     